### Configuration

In [0]:
%python
# Dataset configuration

%python
from pyspark.sql import functions as F
from pyspark.sql.types import *

NUM_CUSTOMERS = 100_000
NUM_PRODUCTS = 1_000
NUM_STORES = 500

# Start small while developing.
# We will increase this later for performance experiments.
NUM_TRANSACTIONS = 5_000_000

SEED = 42

### Generate Customers

In [0]:
%python
customers = (
    spark.range(1, NUM_CUSTOMERS + 1)
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "customer_name",
        F.concat(F.lit("Customer_"), F.col("customer_id"))
    )
    .withColumn(
        "customer_segment",
        F.when(F.col("customer_id") % 10 < 2, "Premium")
         .when(F.col("customer_id") % 10 < 6, "Standard")
         .otherwise("Basic")
    )
    .withColumn(
        "city",
        F.element_at(
            F.array(
                F.lit("Toronto"),
                F.lit("Mississauga"),
                F.lit("Vancouver"),
                F.lit("Calgary"),
                F.lit("Montreal")
            ),
            ((F.col("customer_id") % 5) + 1).cast("int")
        )
    )
    .withColumn(
        "state",
        F.element_at(
            F.array(
                F.lit("ON"),
                F.lit("ON"),
                F.lit("BC"),
                F.lit("AB"),
                F.lit("QC")
            ),
            ((F.col("customer_id") % 5) + 1).cast("int")
        )
    )
)

### Generate Products

In [0]:
%python
products = (
    spark.range(1, NUM_PRODUCTS + 1)
    .withColumnRenamed("id", "product_id")
    .withColumn(
        "product_name",
        F.concat(F.lit("Product_"), F.col("product_id"))
    )
    .withColumn(
        "category",
        F.element_at(
            F.array(
                F.lit("Electronics"),
                F.lit("Grocery"),
                F.lit("Clothing"),
                F.lit("Home"),
                F.lit("Sports")
            ),
            ((F.col("product_id") % 5) + 1).cast("int")
        )
    )
    .withColumn(
        "unit_price",
        F.round(
            F.rand(SEED) * 490 + 10,
            2
        )
    )
)

### Generate Stores

In [0]:
%python
stores = (
    spark.range(1, NUM_STORES + 1)
    .withColumnRenamed("id", "store_id")
    .withColumn(
        "store_name",
        F.concat(F.lit("Store_"), F.col("store_id"))
    )
    .withColumn(
        "city",
        F.element_at(
            F.array(
                F.lit("Toronto"),
                F.lit("Vancouver"),
                F.lit("Calgary"),
                F.lit("Montreal"),
                F.lit("Ottawa")
            ),
            ((F.col("store_id") % 5) + 1).cast("int")
        )
    )
    .withColumn(
        "region",
        F.element_at(
            F.array(
                F.lit("East"),
                F.lit("West"),
                F.lit("West"),
                F.lit("East"),
                F.lit("East")
            ),
            ((F.col("store_id") % 5) + 1).cast("int")
        )
    )
)

### Generate Transactions

In [0]:
%python
transactions = (
    spark.range(1, NUM_TRANSACTIONS + 1)
    .withColumnRenamed("id", "transaction_id")
    
    .withColumn(
        "customer_id",
        (F.rand(SEED) * NUM_CUSTOMERS)
        .cast("long") + F.lit(1)
    )
    
    .withColumn(
        "product_id",
        (F.rand(SEED + 1) * NUM_PRODUCTS)
        .cast("long") + F.lit(1)
    )
    
    .withColumn(
        "store_id",
        (F.rand(SEED + 2) * NUM_STORES)
        .cast("long") + F.lit(1)
    )
    
    .withColumn(
        "transaction_date",
        F.date_add(
            F.to_date(F.lit("2025-01-01")),
            (F.rand(SEED + 3) * 365).cast("int")
        )
    )
    
    .withColumn(
        "quantity",
        (F.rand(SEED + 4) * 5).cast("int") + F.lit(1)
    )
    
    .withColumn(
        "unit_price",
        F.round(
            F.rand(SEED + 5) * 490 + 10,
            2
        )
    )
    
    .withColumn(
        "discount",
        F.round(
            F.rand(SEED + 6) * 0.30,
            2
        )
    )
    
    .withColumn(
        "amount",
        F.round(
            F.col("quantity")
            * F.col("unit_price")
            * (F.lit(1) - F.col("discount")),
            2
        )
    )
)

### Validate all four table

In [0]:
%python
print("Customers:", customers.count())
print("Products:", products.count())
print("Stores:", stores.count())
print("Transactions:", transactions.count())

### Validation

In [0]:
%python
transactions.select(
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date"),
    F.sum("amount").alias("total_revenue"),
    F.avg("amount").alias("avg_transaction")
).show()